# Notebook for causal analysis of the SHAPly results using DoWhy-library

SEM:
SHAP liefert die wichtigen Punkte/Prediktoren, jedoch fehlen die Richtungen des kausalen Zusammenhangs

Für SEM muss man die Richtungen vordefinieren und sie sozusagen damit bestätigen oder wiederlegen
Für die Richtungen -> Theorien/Arbeiten lesen, eventuell DoWhy lib wenn die das kann
Eine Arbeit zu: >STD Theorie >PERMA >Maslow > vielleicht WHO oder WHR

"ADLDOT.1.pdf" in NotebookML und nach Prediktoren/Kausalen Zusammenhängen fragen


FRAGE: Welche Prediktoren zu welcher Kategorie in den Theorien? Sind die 3 Theorien relevant?
Ablauf:
1. Arbeiten suchen und kausale Zusammenhänge definieren (Pfeile)
    2. STD Prediktoren und Kausales raussuchen + 1-2 Arbeiten die stützen, Nebenpfade??
    3. PERMA Prediktoren und Kausales raussuchen + 1-2 Arbeiten die stützen, Nebenpfade??
    4. Maslows Prediktoren und Kausales raussuchen + 1-2 Arbeiten die stützen, Nebenpfade durch die Pyramide nach oben
5. SEM anwenden auf den Datensatz -> Pfadiagramm
6. Optional DoWhy als Validierung


DoWhy

1 Predictor -> {mehrere Confounder}


**Features, die in allen 3 Modellen vorkommen (3× vertreten)**

Health condition
I generally feel that what I do in life is worthwhile
How satisfied with education?
Can't find the way because life has become so complicated?
I feel I am free to decide how to live my life
I am optimistic about the future
Household able to make ends meet?
Personal financial situation
How much trust the police?


**2x**
Deprivation index: No. of items hhold can't afford (RF, XG)
Quality of education system? (RF, XG)
The value of what I do is not recognised by others? (RF, XG)
Marital status (SVR, XG)
Can most people be trusted? (SVR, XG)

**1x**
**Nur RF:**
Can afford to pay for a week's annual holiday away?
Quality of health services?
Feel left out of society?

**Nur SVR:**
Financial situation of your household compared to 12 months ago?
Education completed
Quality of long term care services?

**Nur XGBoost:**
I seldom have time to do the things I really enjoy
Neighbourhood problems – traffic

In [2]:
from dowhy import CausalModel
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd

### Feature list with corresponding confounders

In [3]:
outcome = "How happy are you?"

In [10]:
feature_confounder_map_x3 = {
    "I feel I am free to decide how to live my life": [
        "Income quartiles", "Education completed", "How much trust the government?"
    ],
    "Health condition": [
        "Age", "Income quartiles", "Chronic health problems?"
    ],
    "Can't find the way because life has become so complicated?": [
        "Education completed", "Employment - 7 groups", "A person to get support from when feeling depressed"
    ],
    "How satisfied with education?": [
        "Education completed", "Income quartiles"
    ],
    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "Employment - 7 groups",
        "Income quartiles"
    ],
    "I am optimistic about the future": [
        "Income quartiles",
        "Chronic health problems?",
        "A person to get support from when feeling depressed"
    ],
    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Financial situation of your household compared to 12 months ago?"
    ],
    "Personal financial situation": [
        "Employment - 7 groups",
        "Income quartiles",
        "Household size"
    ],
    "How much trust the police?": [
        "How much trust the government?",
        'Neighbourhood problems - crime, violence or vandalism',
        'How much trust the legal system?'
    ]
}

feature_confounder_map_x2 = {
    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?"
    ],
    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed"
    ],
    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?"
    ],
    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?"
    ]
}

### Run causal analysis using doWhy

In [11]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]

In [12]:
def calculate_causal_values(confounder_map):
    # Save results
    results = []
    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data,
                treatment=treatment,
                outcome=outcome,
                common_causes=confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders": ", ".join(confounders)
            })
        except Exception as e:
            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders": ", ".join(confounders),
                "Error": str(e)
            })
    return results




# Results
results_df = pd.DataFrame(calculate_causal_values(feature_confounder_map_x2))

results_df = pd.DataFrame(calculate_causal_values(feature_confounder_map_x3))
results_df.head(10)

Treatment: Deprivation index: No. of items hhold can't afford
Treatment: The value of what I do is not recognised by others?
Treatment: Can most people be trusted?
Treatment: Marital status
Treatment: I feel I am free to decide how to live my life
Treatment: Health condition
Treatment: Can't find the way because life has become so complicated?
Treatment: How satisfied with education?
Treatment: I generally feel that what I do in life is worthwhile
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders
0,I feel I am free to decide how to live my life,0.499720,"Income quartiles, Education completed, How muc..."
1,Health condition,0.671812,"Age, Income quartiles, Chronic health problems?"
2,Can't find the way because life has become so ...,-0.441010,"Education completed, Employment - 7 groups, A ..."
3,How satisfied with education?,0.197422,"Education completed, Income quartiles"
4,I generally feel that what I do in life is wor...,0.693301,A person to get support from when feeling depr...
5,I am optimistic about the future,0.493040,"Income quartiles, Chronic health problems?, A ..."
6,Household able to make ends meet?,0.410742,"Employment - 7 groups, Income quartiles, Finan..."
7,Personal financial situation,0.574782,"Employment - 7 groups, Income quartiles, House..."
8,How much trust the police?,0.109481,"How much trust the government?, Neighbourhood ..."
